# Automatické tréningy pre rôzne n_input a horizonty (DST+1 … DST+6) – Transformer

Tento notebook načíta dáta **iba raz** a potom postupne vytvorí a natrénuje samostatný model pre každú kombináciu:
- `n_input` ∈ {6, 12, 18, 24, 30, 36, 42, 48}
- `y_col` = `DST+1` … `DST+6`

Postup ostáva rovnaký ako v notebooku `DST+1_model_training`, ale namiesto **LSTM** sa používa **Transformer encoder**.
Model/weighty sa ukladajú do `models/` a súhrn metrík do `results/summary.csv`.


In [1]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras import layers

# Reprodukovateľnosť (voliteľné)
SEED = 42
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)
tf.random.set_seed(SEED)


2026-03-07 21:38:56.942934: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
import os; 
print(os.getcwd())

/home/jovyan/data/lightning/TereziaD/DP_pokracovanieBP/BP_Drengubiakova_Terezia/3_modelovanie/train automate


In [3]:
# =========================
# Nastavenia
# =========================

# Cesty k datasetom (ponechané ako v pôvodnom notebooku)
BASE_PATH = "/home/jovyan/data/lightning/TereziaD/DP_pokracovanieBP/BP_Drengubiakova_Terezia/0_datasety/"
file_path_train = BASE_PATH + "train_omni.csv"
file_path_test  = BASE_PATH + "test_omni.csv"


# Tréningové nastavenia
BATCH_SIZE = 256
EPOCHS = 200
PATIENCE = 25  # early stopping na val_mae

# Kombinácie na tréning
N_INPUT_LIST = [6, 12, 18, 24, 30, 36, 42, 48]
HORIZONS = [4]

# Výstupné priečinky
MODELS_DIR = "models_transformer"
RESULTS_DIR = "results_transformer"
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)


In [4]:
# =========================
# Načítanie dát (iba raz)
# =========================
train_raw = pd.read_csv(file_path_train)
test_raw  = pd.read_csv(file_path_test)

# Skontroluj, aké DST+ stĺpce existujú (pomôže odhaliť preklepy v názvoch)
dst_cols = [c for c in train_raw.columns if c.startswith("DST+")]
print("DST+ stĺpce v train:", dst_cols)
print("DST+ stĺpce v test :", [c for c in test_raw.columns if c.startswith("DST+")])

# čas
if "time1" in train_raw.columns:
    train_raw["time1"] = pd.to_datetime(train_raw["time1"])
if "time1" in test_raw.columns:
    test_raw["time1"] = pd.to_datetime(test_raw["time1"])


DST+ stĺpce v train: ['DST+1', 'DST+2', 'DST+3', 'DST+4', 'DST+5', 'DST+6']
DST+ stĺpce v test : ['DST+1', 'DST+2', 'DST+3', 'DST+4', 'DST+5', 'DST+6']


In [5]:
class PositionalEmbedding(layers.Layer):
    """Jednoduché naučiteľné pozičné embeddingy pre sekvenciu dĺžky n_input."""

    def __init__(self, sequence_length: int, d_model: int, **kwargs):
        super().__init__(**kwargs)
        self.sequence_length = sequence_length
        self.d_model = d_model
        self.input_projection = layers.Dense(d_model)
        self.position_embedding = layers.Embedding(input_dim=sequence_length, output_dim=d_model)

    def call(self, inputs):
        positions = tf.range(start=0, limit=self.sequence_length, delta=1)
        x = self.input_projection(inputs)
        pos = self.position_embedding(positions)
        return x + pos

    def get_config(self):
        config = super().get_config()
        config.update({
            "sequence_length": self.sequence_length,
            "d_model": self.d_model,
        })
        return config


class TransformerEncoder(layers.Layer):
    """Transformer encoder blok pre časové rady."""

    def __init__(self, d_model: int, num_heads: int, ff_dim: int, dropout: float = 0.1, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        self.dropout = dropout

        self.attention = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads, dropout=dropout)
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dropout(dropout),
            layers.Dense(d_model),
        ])
        self.norm_1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm_2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout_1 = layers.Dropout(dropout)
        self.dropout_2 = layers.Dropout(dropout)

    def call(self, inputs, training=False):
        attn_output = self.attention(inputs, inputs, training=training)
        attn_output = self.dropout_1(attn_output, training=training)
        x = self.norm_1(inputs + attn_output)

        ffn_output = self.ffn(x, training=training)
        ffn_output = self.dropout_2(ffn_output, training=training)
        return self.norm_2(x + ffn_output)

    def get_config(self):
        config = super().get_config()
        config.update({
            "d_model": self.d_model,
            "num_heads": self.num_heads,
            "ff_dim": self.ff_dim,
            "dropout": self.dropout,
        })
        return config


def build_model(n_input: int) -> keras.Model:
    """Transformer architektúra"""
    d_model = 64
    num_heads = 4
    ff_dim = 128
    dropout = 0.1

    inputs = keras.Input(shape=(n_input, 1))
    x = PositionalEmbedding(sequence_length=n_input, d_model=d_model)(inputs)
    x = TransformerEncoder(d_model=d_model, num_heads=num_heads, ff_dim=ff_dim, dropout=dropout)(x)
    x = TransformerEncoder(d_model=d_model, num_heads=num_heads, ff_dim=ff_dim, dropout=dropout)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(64, activation="relu")(x)
    outputs = layers.Dense(1, activation="linear")(x)

    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(loss="mse", optimizer=keras.optimizers.Adam(learning_rate=1e-3), metrics=["mae"])
    return model


def make_splits(train_df: pd.DataFrame, test_df: pd.DataFrame, y_col: str):
    """Pripraví train/valid/test dáta pre daný y_col (bez NaN)."""

    features = ["DST", y_col]

    train = train_df[features].copy()
    test  = test_df[features].copy()

    # 🔧 ODSTRÁNENIE NaN spôsobených DST+X
    train = train.dropna().reset_index(drop=True)
    test  = test.dropna().reset_index(drop=True)

    # split train/valid (časovo korektný)
    valid_size = int(len(train) * 0.2)

    if valid_size == 0:
        raise ValueError(f"Príliš málo dát po dropna pre {y_col}")

    valid = train.iloc[-valid_size:, :].copy()
    train = train.iloc[:-valid_size, :].copy()

    X_train = train["DST"].values
    y_train = train[y_col].values

    X_val = valid["DST"].values
    y_val = valid[y_col].values

    X_test = test["DST"].values
    y_test = test[y_col].values

    return (X_train, y_train), (X_val, y_val), (X_test, y_test)


In [6]:
def train_one(y_col: str, n_input: int):
    (X_train, y_train), (X_val, y_val), (X_test, y_test) = make_splits(train_raw, test_raw, y_col)

    train_gen = TimeseriesGenerator(X_train, y_train, length=n_input, batch_size=BATCH_SIZE)
    val_gen   = TimeseriesGenerator(X_val, y_val, length=n_input, batch_size=BATCH_SIZE)
    test_gen  = TimeseriesGenerator(X_test, y_test, length=n_input, batch_size=BATCH_SIZE)

    if len(train_gen) == 0 or len(val_gen) == 0:
        print(f"[SKIP] {y_col}, n_input={n_input} – prázdny generátor")
        return None

    
    model = build_model(n_input)

    model_path = os.path.join(MODELS_DIR, f"{y_col}_{n_input}H.keras")
    checkpoint = ModelCheckpoint(model_path, monitor="val_mae", verbose=1, save_best_only=True, mode="min")
    early = EarlyStopping(monitor="val_mae", mode="min", patience=PATIENCE, restore_best_weights=True)
    callbacks = [checkpoint, early]

    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=EPOCHS,
        verbose=1,
        callbacks=callbacks
    )

    # finálne vyhodnotenie na test sete
    test_loss, test_mae = model.evaluate(test_gen, verbose=0)

    # uložiť aj history (voliteľné)
    hist_df = pd.DataFrame(history.history)
    hist_csv = os.path.join(RESULTS_DIR, f"history_{y_col}_{n_input}H.csv")
    hist_df.to_csv(hist_csv, index=False)

    return {
        "y_col": y_col,
        "horizon_hours": int(y_col.split("+")[1]),
        "n_input": n_input,
        "best_val_mae": float(np.min(hist_df["val_mae"])) if "val_mae" in hist_df else np.nan,
        "best_val_loss": float(np.min(hist_df["val_loss"])) if "val_loss" in hist_df else np.nan,
        "test_mae": float(test_mae),
        "test_loss": float(test_loss),
        "model_path": model_path,
        "history_path": hist_csv,
        "epochs_ran": int(len(hist_df)),
    }


In [ ]:
# =========================
# Spustenie všetkých tréningov
# =========================
summary_rows = []

for h in HORIZONS:
    y_col = f"DST+{h}"

    # ochrana, ak by stĺpec neexistoval
    if y_col not in train_raw.columns or y_col not in test_raw.columns:
        print(f"[SKIP] {y_col} neexistuje v datasete.")
        continue

    for n_input in N_INPUT_LIST:
        print("\n" + "="*80)
        print(f"Trénujem: y_col={y_col}, n_input={n_input}")
        print("="*80)

        row = train_one(y_col, n_input)
        if row is not None:
            summary_rows.append(row)


summary = pd.DataFrame(summary_rows)
summary_path = os.path.join(RESULTS_DIR, "summary4.csv")
summary.to_csv(summary_path, index=False)

summary.sort_values(["horizon_hours", "n_input"]).head(20), summary_path



Trénujem: y_col=DST+4, n_input=6
Epoch 1/200


/opt/conda/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


264/896 ━━━━━━━━━━━━━━━━━━━━ 1:10 112ms/step - loss: 418.1353 - mae: 13.0991

## Poznámky
- Postup, dátové splity, generátory, checkpointy aj export výsledkov ostávajú rovnaké ako v LSTM verzii.
- Rozdiel je iba v architektúre `build_model`, kde sa používa 2× Transformer encoder s pozičnými embeddingami.
- Ak chceš trénovať všetky kombinácie, stačí v bunke **Nastavenia** odkomentovať celý zoznam `N_INPUT_LIST` a prípadne nastaviť `HORIZONS = [1, 2, 3, 4, 5, 6]`.
